# Chapter 2 — Meaning Becomes Geometry

**Book alignment:** Embeddings From First Principles, Chapter 2

**Notebook role:** `SYNTHETIC_DEMO` — constructs a controlled toy example of the chapter's mechanism. It does **not** reproduce any measured benchmark; it shows the effect is real and inspectable.

**Question this notebook isolates:** Once information is a vector, do semantic questions
become geometric ones (distance, direction, angle, cluster) — and where does that
translation leak? Concretely: does a 2-D projection of a real embedding space *invent*
separations the geometry does not have?

Self-contained NumPy. The RELATE projection numbers the chapter reports (top-2 PCs keep
~15% variance, ~20% of neighbourhoods survive) are cited in prose; the mechanism is
reproduced here on synthetic data.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# A space you can draw: 6 words, 2 hand-assigned coordinates (royalty/power, gender).
W = {"king": (0.9, 0.8), "queen": (0.9, -0.8), "man": (0.1, 0.9),
     "woman": (0.1, -0.9), "apple": (-0.8, 0.0), "orange": (-0.8, 0.1)}
names = list(W)
X = np.array([W[n] for n in names], float)

## 1. Every semantic question has a geometric form

In [ ]:
def vec(w):
    return X[names.index(w)]

def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

# relatedness -> distance
d_kq = np.linalg.norm(vec("king") - vec("queen"))
d_ka = np.linalg.norm(vec("king") - vec("apple"))
print(f"dist(king, queen) = {d_kq:.2f}   dist(king, apple) = {d_ka:.2f}")
assert d_kq < d_ka

# contrast -> direction; the same semantic change is (locally) the same direction
gender_royalty = vec("king") - vec("queen")
gender_person  = vec("man") - vec("woman")
print(f"king-queen direction {gender_royalty.round(2)}   man-woman {gender_person.round(2)}")
assert cos(gender_royalty, gender_person) > 0.9

# category -> cluster: fruit tight, far from royalty
fruit = np.array([vec("apple"), vec("orange")])
royal = np.array([vec("king"), vec("queen")])
assert np.linalg.norm(fruit[0] - fruit[1]) < np.linalg.norm(fruit.mean(0) - royal.mean(0))
print("relatedness->distance, contrast->direction, orientation->angle, category->cluster")

## 2. Analogy arithmetic works — but only locally, and partly by curation

In [ ]:
guess = vec("king") - vec("man") + vec("woman")
nn = min((n for n in names if n != "king"),
         key=lambda n: np.linalg.norm(vec(n) - guess))
print(f"king - man + woman  ->  nearest word: {nn}")
assert nn == "queen"
# in this hand-built space the gender direction is globally constant; in a learned space
# it drifts by region, and the famous examples are partly picked. (chapter section 'Where the translation leaks')
print("true here by construction; a learned space only gives this on average, locally")

## 3. A 2-D projection invents structure the data does not have

Three well-separated blobs in 50-D plus uniform noise. Project to 2-D with PCA and measure
how much of each point's 10-NN neighbourhood survives.

In [ ]:
d = 50
centers = rng.standard_normal((12, d))
centers /= np.linalg.norm(centers, axis=1, keepdims=True)
centers *= 3.0                                       # 12 clusters spread over a 50-d sphere
data = np.vstack([c + rng.standard_normal((40, d)) * 0.9 for c in centers])

def knn(M, k=10):
    D = ((M[:, None] - M[None]) ** 2).sum(-1)
    return np.argsort(D, axis=1)[:, 1:k + 1]

Xc = data - data.mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
proj = Xc @ Vt[:2].T
var_kept = (S[:2] ** 2).sum() / (S ** 2).sum()

overlap = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(knn(data), knn(proj))])
print(f"variance kept by top 2 PCs : {var_kept:.0%}")
print(f"10-NN neighbourhood survival: {overlap:.0%}")
assert var_kept < 0.30 and overlap < 0.80
# 12 clusters cannot all be separated in a 2-d plane: some land on top of each other
d2 = ((proj[:, None] - proj[None]) ** 2).sum(-1)
collisions = sum(1 for i in range(12) for j in range(i + 1, 12)
                 if np.linalg.norm(proj[i * 40:(i + 1) * 40].mean(0)
                                   - proj[j * 40:(j + 1) * 40].mean(0)) < 1.5)
print(f"cluster-centre pairs the plot fuses together: {collisions}")
print("\nthe picture is a transformation of a transformation - use it for hypotheses, not proof")

## 4. Lab 2 replay — build one, then break one (measured)

The chapter's Lab 2 ran for real: 12 words in four hand-built clusters, embedded with
`bge-small-en-v1.5`, ranked in the full 384-d space vs PCA-2D and t-SNE-2D. This cell
reloads the frozen artifact and checks the book's table. UMAP was deliberately not run
(dependency not vendored) — the chapter reports PCA + t-SNE as measured.

**Try it yourself:** `WORDS`, `MODEL`, `SEED` in
`experiments/embeddings-from-first-principles/wave1/lab02_break_one.py` — replace the words,
swap in `mpnet-base`, vary the t-SNE perplexity/seed, add an ambiguous word.

In [ ]:
from pathlib import Path
import json

root = Path.cwd()
while not (root / "experiments" / "embeddings-from-first-principles").exists():
    root = root.parent
art = root / "experiments" / "embeddings-from-first-principles" / "wave1" / "artifacts"

lab = json.loads((art / "lab02-break-one.json").read_text())
pca = json.loads((art / "pca-projection.json").read_text())

# Part A: the hand-built geometry means what we chose
assert lab["part_a_handbuilt"]["holds"]  # within-max 0.412 < across-min 4.601
print("hand-built: within-max", lab["part_a_handbuilt"]["within_max"],
      "< across-min", lab["part_a_handbuilt"]["across_min"])

# Part B: the changing relationship — cat<->car falls off a cliff in 2D
rows = {r["pair"]: r for r in lab["pairs"]}
assert (rows["cat <-> car"]["full_rank"], rows["cat <-> car"]["pca_rank"],
        rows["cat <-> car"]["tsne_rank"]) == (3, 10, 7)
for p, r in rows.items():
    print(f"{p:14s} full={r['full_rank']:2d}  pca={r['pca_rank']:2d}  tsne={r['tsne_rank']:2d}")
print("top-3 preserved: PCA", lab["preserved_top3"]["pca"],
      " t-SNE", lab["preserved_top3"]["tsne"])

# At RELATE scale the damage is larger
assert abs(pca["results"]["variance_retained_top2_pcs"] - 0.1531) < 1e-9
assert abs(pca["results"]["knn10_overlap_full_vs_2d"] - 0.2032) < 1e-9
print("RELATE-250: variance kept", pca["results"]["variance_retained_top2_pcs"],
      " 10-NN survive", pca["results"]["knn10_overlap_full_vs_2d"])
print("\nthe picture changed; the embedding did not")

## What we earned

Semantic questions really do become geometric ones — relatedness↔distance,
contrast↔direction, category↔cluster. But a 2-D picture of an embedding space is *another*
lossy transformation: on synthetic blobs it kept ~20% of the variance and lost ~half of
every neighbourhood (the book measured 15% / 24% on real RELATE embeddings), and it hands
structureless noise a crisp position. Verify structural claims against the space you query,
never the plot.

**Notebook 03 / Chapter 3** learns a small space from scratch — counts, then factorisation —
so embedding structure is visibly *learned compression*, not coordinates handed down.